17a. Chạy thử toàn bộ quy trình test trên VAL, CPU (GĐ H, trước tag freeze-v1; không đọc test)

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

ALLOW_GPU = False                  # toàn bộ quy trình chạy được trên CPU (suy luận và huấn luyện baseline cây); không dùng GPU
SPLIT = "val"                # "val" = chạy thử trước khi đóng băng; "test" = chạy test một lần
REF = ""                    # "" = nhánh main (chạy thử); "freeze-v1" = bản đóng băng
WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
URL = "https://github.com/minucedev/gnn-iot-ids.git"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None, "split": SPLIT, "ref": REF or "main"}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), log=None, **kw):
    """Chạy lệnh; có `log` thì ghi toàn bộ đầu ra vào file đó (in 6000 ký tự cuối), không thì in 9000 ký tự cuối."""
    if log:
        with open(log, "w", encoding="utf-8") as f:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True, **kw)
        print(open(log, encoding="utf-8").read()[-6000:])
    else:
        r = subprocess.run(cmd, capture_output=True, text=True, **kw)
        print(r.stdout[-9000:], end="")
        print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    """Đường dẫn đầu tiên khớp glob đệ quy dưới /kaggle/input (báo lỗi rõ nếu không có)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", *(["--branch", REF] if REF else []), URL, REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    if REF:                                         # commit đã clone phải đúng commit của tag trên GitHub
        lines = subprocess.run(["git", "ls-remote", URL, f"refs/tags/{REF}", f"refs/tags/{REF}^{{}}"], capture_output=True, text=True).stdout.split("\n")
        tag_sha = [l.split()[0] for l in lines if l.strip()][-1]
        assert tag_sha == INFO["commit"], f"Commit đã clone {INFO['commit']} khác commit của tag {REF}: {tag_sha}"
        INFO["tag"], INFO["tag_commit"] = REF, tag_sha
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| tag:", INFO.get("tag"), "| máy:", INFO["device"], INFO["gpu"], "| split:", SPLIT)
assert ALLOW_GPU or INFO["device"] == "CPU", "Không dùng GPU cho notebook này: tắt GPU trong cài đặt notebook"

## pytest toàn bộ (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    # test tái lập hai lần chạy độc lập không ổn định trên GPU (scatter không tất định); nó đã pass trên CPU ở notebook 16t và 16a
    r = run([sys.executable, "-m", "pytest", "-q", "--deselect", "tests/test_hybrid.py::test_run_hybrid_end_to_end_both_modes_val_only_reproducible"] if INFO["device"] == "GPU" else [sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## Đường dẫn đầu vào
Đồ thị: output của 04 (ToN) và 11 (BoT). Checkpoint GNN: 07 (ToN) và 14 (BoT). Hybrid chính: 15f (ToN), 15e (BoT). LightGBM của hybrid nhẹ: 16d (ToN).
Ngưỡng, cấu hình baseline và cấu hình (c) lấy từ bản repo tại commit đang chạy (đã đóng băng cùng tag).

In [ ]:
DS_T, DS_B = "nf_ton_iot_v3", "nf_bot_iot_v3"
OLD = f"{REPO}/reports/kaggle"
SPEC = {
    DS_T: dict(data=os.path.dirname(find(f"{DS_T}/split.json")),
               ckpt=os.path.dirname(find(f"ckpt/{DS_T}_gat_s42.pt")),
               folds=os.path.dirname(os.path.dirname(find(f"models_hybrid/{DS_T}_s42/lgbm_oofavg_il.pkl"))),
               light=os.path.dirname(os.path.dirname(find(f"models_light/{DS_T}_s42/lgbm_n100_l15.pkl"))),
               baseline_search=f"{OLD}/gnn-iot-ids-05-baselines/baseline_search_{DS_T}.json",
               ablation_search=f"{OLD}/gnn-iot-ids-16d-ablation-ton/ablation_search_{DS_T}.json",
               gnn_csv=f"{OLD}/gnn-iot-ids-07-train-gnn/results_gnn_{DS_T}.csv",
               hybrid_csv=f"{OLD}/gnn-iot-ids-15f-hybrid-il-ton/results_hybrid_oofavg_il_{DS_T}.csv", cross_to=DS_B),
    DS_B: dict(data=os.path.dirname(find(f"{DS_B}/split.json")),
               ckpt=os.path.dirname(find(f"ckpt/{DS_B}_gat_s42.pt")),
               folds=os.path.dirname(os.path.dirname(find(f"models_hybrid/{DS_B}_s42/lgbm_oofavg_il.pkl"))),
               light=None,
               baseline_search=f"{OLD}/gnn-iot-ids-13-baselines/baseline_search_{DS_B}.json",
               ablation_search=f"{OLD}/gnn-iot-ids-16a-ablation-bot/ablation_search_{DS_B}.json",
               gnn_csv=f"{OLD}/gnn-iot-ids-14-train-gnn-bot/results_gnn_{DS_B}.csv",
               hybrid_csv=f"{OLD}/gnn-iot-ids-15e-hybrid-il/results_hybrid_oofavg_il_{DS_B}.csv", cross_to=DS_T),
}
CFG = dict(split=SPLIT, out=WORK, cross=True, expect_commit=INFO["commit"][:7],
           thresholds=f"{OLD}/gnn-iot-ids-16c-aggregate/thresholds.json",
           thresholds_c={ds: f"{OLD}/gnn-iot-ids-16h-c-threshold/thresholds_c_{ds}.json" for ds in SPEC}, datasets=SPEC)
for ds, s in SPEC.items():
    for k, v in s.items():
        if v and k != "cross_to":
            assert os.path.exists(v), f"{ds}.{k}: không thấy {v}"
for p in (CFG["thresholds"], *CFG["thresholds_c"].values()):
    assert os.path.exists(p), f"Không thấy {p}"
json.dump(CFG, open(f"{WORK}/final_test_config.json", "w"), ensure_ascii=False, indent=1)
INFO["config"] = CFG
save_info()
print(json.dumps(CFG, indent=1, ensure_ascii=False))

## Chạy mọi mô hình, 2 bộ, 3 seed
`scripts/final_test.py` (docstring mô tả đầy đủ): với mỗi (bộ, mô hình, seed) dựng mô hình, kiểm F1 val khớp, rồi đánh giá `SPLIT` đúng một lần ở hai chế độ
(argmax; ngưỡng đã đóng băng, hậu tố `_thr`), ghi chỉ số theo tầng, dự đoán từng node và thử chéo giữa 2 bộ.

In [ ]:
with step(f"final_test_{SPLIT}"):
    run([sys.executable, "-I", "scripts/final_test.py", f"{WORK}/final_test_config.json"], log=f"{WORK}/final_test_log.txt", cwd=REPO)

## Kiểm tra đầu ra và run_info.json

In [ ]:
import csv, pandas as pd
rows = []
for ds in SPEC:
    rows += list(csv.DictReader(open(f"{WORK}/results_{SPLIT}_{ds}.csv", encoding="utf-8")))
df = pd.DataFrame(rows)
n_in = {ds: int((df["dataset"] == ds).sum()) for ds in SPEC}
n_cross = {d: int((df["dataset"] == d).sum()) for d in df["dataset"].unique() if "→" in d}
led = pd.read_csv(f"{WORK}/ledger_{SPLIT}.csv")
INFO["check"] = {
    "rows_in_dataset": n_in, "rows_cross": n_cross, "n_ledger": len(led), "n_ledger_unique": int(len(led.drop_duplicates(["dataset", "model", "seed", "split"]))),
    "commits_in_results": sorted(df["commit"].unique().tolist()), "splits_in_results": sorted(df["split"].unique().tolist()),
    "val_f1_diff_max_abs": float(led["val_f1_diff"].abs().max()), "ts_first": led["ts_utc"].min(), "ts_last": led["ts_utc"].max(),
}
assert INFO["check"]["n_ledger"] == INFO["check"]["n_ledger_unique"], "có mục chạy hai lần"
assert df["split"].nunique() == 1 and df["split"].iloc[0] == SPLIT
assert df["commit"].nunique() == 1 and df["commit"].iloc[0] == INFO["commit"][:7], (df["commit"].unique(), INFO["commit"])
assert df.shape[1] == 20
save_info()
print(json.dumps(INFO["check"], indent=1, ensure_ascii=False))

In [ ]:
where = ("tag `" + REF + "`") if REF else "nhánh main (chạy thử trên val, trước khi đóng băng)"
nb_name = "gnn-iot-ids-17-final-test" if SPLIT == "test" else "gnn-iot-ids-17a-dryrun-val"
readme = f"""# Kết quả chạy `{SPLIT}` của GĐ H (notebook {nb_name})

Repo `minucedev/gnn-iot-ids` tại {where}, commit `{INFO['commit']}`.

## Quy ước tên mô hình trong `results_{SPLIT}_<bộ>.csv`

- Mỗi (bộ, mô hình, seed) chạy `{SPLIT}` đúng **một lần**; từ cùng một lần dự đoán ghi **hai dòng**:
  - tên mô hình giữ nguyên (ví dụ `lightgbm`): chế độ **argmax** (lớp có xác suất cao nhất);
  - tên mô hình thêm hậu tố `_thr` (ví dụ `lightgbm_thr`): chế độ **ngưỡng đã hiệu chỉnh** trên val (đóng băng ở `reports/kaggle/gnn-iot-ids-16c-aggregate/thresholds.json`,
    riêng `lgbm_neighbor_mean` ở `reports/kaggle/gnn-iot-ids-16h-c-threshold/`): điểm tấn công = 1 - P(Normal); điểm > ngưỡng thì lớp tấn công có xác suất cao nhất, ngược lại Normal.
- Dòng thử chéo (task 5.6) có cột `dataset` dạng `A→B` (mô hình huấn luyện trên A, đánh giá trên B); chỉ giữ node B thuộc lớp chung (Normal, DoS_DDoS, Truy_cap_trai_phep),
  chỉ xét các lớp chung khi dự đoán; đặc trưng node của B được chuẩn hóa lại theo scaler của A. Mô hình: hybrid_gat_lgbm_oofavg_il, gat, lightgbm, lgbm_neighbor_mean.
- `hybrid_gat_lgbm_oofavg_il` = hybrid chính (5 GAT nhóm + LightGBM); `hybrid_light` = hybrid nhẹ (chỉ NF-ToN-IoT-v3: 2 GAT nhóm + LightGBM n=100, l=15). Ở NF-BoT-IoT-v3 hybrid nhẹ
  trùng hybrid chính nên không có dòng `hybrid_light` riêng.
- `n_params`: baseline cây và `lgbm_neighbor_mean` đếm theo `evaluate.count_params` (tổng số node của mọi cây); hybrid = tham số các GAT + số lá LightGBM (như GĐ F).
  `train_time_s` của GNN và hybrid là thời gian huấn luyện lúc đó (đọc từ CSV val cũ), của baseline là thời gian huấn luyện lại; `infer_ms_per_window` là p95, CPU 1 luồng.
- `tiers_{SPLIT}_<bộ>.csv`: chỉ số theo tầng: `all`, `initiator` (node có khởi tạo flow, n_out > 0), `receive_only` (node chỉ nhận flow).
- `pred_<bộ>_<mô hình>_s<seed>.parquet`: dự đoán từng node (cột `y_true`, `pred` = argmax, `pred_thr`, `score` = 1 - P(Normal)), cùng thứ tự với `nodes_<bộ>_{SPLIT}.parquet`.
  Chỉ để trên Kaggle (không tải về).
- `val_check_{SPLIT}_<bộ>.json`: F1-macro val dựng lại so với F1 val lúc hiệu chỉnh ngưỡng (dừng nếu lệch > 5e-4). `ledger_{SPLIT}.csv`: thời điểm (UTC) chạy từng mục.
"""
open(f"{WORK}/README.md", "w", encoding="utf-8").write(readme)
print(open(f"{WORK}/run_info.json", encoding="utf-8").read()[-3500:])